In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv("mobile_usage_dataset.csv")
df.head(10)


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum()/len(df))*100

In [ ]:
df['Device_Brand'] =df['Device_Brand'].fillna(df['Device_Brand'].mode()[0])

In [ ]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

In [ ]:
#styling
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11
})

In [ ]:
df['Age'] =df['Age'].fillna(df['Age'].median())
df['Age'].round().astype('Int64')



In [ ]:
df['Social_Media_Hours'] =df['Social_Media_Hours'].fillna(df['Social_Media_Hours'].median())


In [ ]:
df['Work_Hours'] =df['Work_Hours'].fillna(df['Work_Hours'].median())


In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
num_cols = ['Age','Social_Media_Hours','Gaming_Hours','Work_Hours','Streaming_Hours','Total_Screen_Time','High_Usage']

plt.figure(figsize=(8,8))

for col in num_cols:
    sns.boxplot(x=df[col])
    plt.title(f'Box plot of {col}')
    plt.show()

In [ ]:
df[num_cols].describe() #to check max-min values and logically impossible ones.

In [ ]:
df['Social_Media_Hours'] = np.where(df['Social_Media_Hours']>24,24,df['Social_Media_Hours']) #capping the max value

In [ ]:
df['Social_Media_Hours'].max() #to check the max value

In [ ]:
#checks co-linearity b/w numeric values (data), hence checking linear dependence

plt.figure(figsize=(8,5))
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm',fmt='.2f')
plt.title("Correlation matrix")
plt.show()

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Total_Screen_Time"], bins=30, kde=True, color="#4C72B0")
plt.title("Distribution of Total Screen Time")
plt.xlabel("Hours per day")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["Age"], bins=20, kde=True, color="#55A868")
plt.title("Age Distribution of Users")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
brand_counts = df["Device_Brand"].value_counts().head(8)

plt.figure(figsize=(10, 6))
sns.barplot(x=brand_counts.index, y=brand_counts.values, palette="viridis")
plt.title("Top Device Brands in Dataset")
plt.xlabel("Brand")
plt.ylabel("Number of users")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
usage_counts = df["High_Usage"].map({0: "Low Usage", 1: "High Usage"}).value_counts()

plt.figure(figsize=(8, 6))
sns.barplot(x=usage_counts.index, y=usage_counts.values, palette=["#95a5a6", "#e74c3c"])
plt.title("High Usage vs Low Usage Users")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="Social_Media_Hours",
    y="Total_Screen_Time",
    hue="High_Usage",
    alpha=0.6,
    palette={0: "#3498db", 1: "#e67e22"}
)
plt.title("Social Media Hours vs Total Screen Time")
plt.tight_layout()
plt.show()

In [ ]:
#removing the duplicate casing issues

df['Gender'] = df['Gender'].str.lower()
df['Device_Brand'] = df['Device_Brand'].str.lower()
df['Location_Type'] = df['Location_Type'].str.lower()


In [ ]:
#regression pipeline

df_encoded = pd.get_dummies(df,drop_first=True) #getting copy of the data frame into a variable for computation.
df_encoded.head(10)

In [ ]:
df.info()
df_encoded.shape

In [ ]:
#declaring x and y variables 
y = df_encoded['Total_Screen_Time'] #target variable

x = df_encoded.drop(['Total_Screen_Time','High_Usage'],axis=1)

print("x shape: ", x.shape)
print("y shape: ", y.shape)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print('x_train shape: ',x_train.shape)
print('x_test shape: ',x_test.shape)
print('y_train shape: ',y_train.shape)
print('y_test shape: ',y_test.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

type(x_train_scaled)
type(x_test_scaled)





In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(x_train_scaled,y_train)
print("model trained")

In [ ]:
y_pred = model.predict(x_test_scaled)
print(y_pred[:5])

In [ ]:
print("Actual values: ",y_test[:5].values)

print("Predicted values: ",y_pred[:5])


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test,y_pred)
mse = np.sqrt(mean_squared_error(y_test,y_pred))
rscore = r2_score(y_test,y_pred)

print("MAE: ",mae)
print("MSE: ",mse)
print("RSCORE: ",rscore)

#0.80-r2score(very strong-nearly 80% accuracy)

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.6, color="#2ecc71", edgecolor="white")
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual Total Screen Time")
plt.ylabel("Predicted Total Screen Time")
plt.title(f"Regression Results: Actual vs Predicted (R² = {rscore:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.6, color="#9b59b6")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted values")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot — Linear Regression")
plt.tight_layout()
plt.show()

In [ ]:
metrics = {"MAE": mae, "RMSE": mse, "R² Score": rscore}

plt.figure(figsize=(8, 5))
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), palette="muted")
plt.title("Linear Regression Performance Metrics")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

In [ ]:
df_encoded.info